# 05 - SQL Analytics

## Customer Intelligence Platform

This notebook demonstrates SQL proficiency by creating a SQLite database
from the cleaned dataset and executing 25+ business-oriented queries.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import sqlite3
from src.load_data import load_clean
from src.config import TARGET


In [ ]:
df = load_clean()


## 1. Create SQLite Database


In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("customers", conn, index=False, if_exists="replace")
print(f"Created SQLite database with {len(df):,} records")


In [ ]:
def run_query(query, description=""):
    """Run a SQL query and display results."""
    if description:
        print(f"\n{'='*60}")
        print(f" {description}")
        print(f"{'='*60}")
    result = pd.read_sql_query(query, conn)
    return result


## 2. Overview Queries


In [ ]:
run_query('''
SELECT
    COUNT(*) AS total_customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
''', "Overall Churn Rate")


In [ ]:
run_query('''
SELECT
    ROUND(SUM("Monthly Charges"), 2) AS total_monthly_revenue,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge,
    ROUND(SUM("Total Charges"), 2) AS total_lifetime_revenue,
    ROUND(AVG("Total Charges"), 2) AS avg_total_charge
FROM customers
''', "Revenue Summary")


## 3. Demographic Queries


In [ ]:
run_query('''
SELECT
    "Senior Citizen",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY "Senior Citizen"
ORDER BY churn_rate_pct DESC
''', "Churn by Senior Citizen Status")


In [ ]:
run_query('''
SELECT
    CASE
        WHEN "Partner" = 'Yes' AND "Dependents" = 'Yes' THEN 'Full Family'
        WHEN "Partner" = 'Yes' AND "Dependents" = 'No' THEN 'Partner Only'
        WHEN "Partner" = 'No' AND "Dependents" = 'Yes' THEN 'Single Parent'
        ELSE 'Single'
    END AS family_status,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge
FROM customers
GROUP BY family_status
ORDER BY churn_rate_pct DESC
''', "Family Status Impact on Churn")


## 4. Service Queries


In [ ]:
run_query('''
SELECT
    "Internet Service",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge
FROM customers
GROUP BY "Internet Service"
ORDER BY churn_rate_pct DESC
''', "Churn by Internet Service Type")


In [ ]:
run_query('''
SELECT
    CASE
        WHEN "Internet Service" = 'Fiber optic' AND "Online Security" = 'No' THEN 'Fiber, No Security'
        WHEN "Internet Service" = 'Fiber optic' AND "Online Security" = 'Yes' THEN 'Fiber, With Security'
        WHEN "Internet Service" = 'DSL' THEN 'DSL'
        ELSE 'No Internet'
    END AS service_profile,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY service_profile
ORDER BY churn_rate_pct DESC
''', "Fiber Security Gap Analysis")


In [ ]:
run_query('''
SELECT
    service_count,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM (
    SELECT *,
        (CASE WHEN "Phone Service" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Multiple Lines" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Online Security" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Online Backup" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Device Protection" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Tech Support" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Streaming TV" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Streaming Movies" = 'Yes' THEN 1 ELSE 0 END) AS service_count
    FROM customers
) subq
GROUP BY service_count
ORDER BY service_count
''', "Service Adoption vs Churn")


## 5. Contract & Payment Queries


In [ ]:
run_query('''
SELECT
    "Contract",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(SUM("Monthly Charges"), 2) AS total_monthly_revenue
FROM customers
GROUP BY "Contract"
ORDER BY churn_rate_pct DESC
''', "Revenue by Contract Type")


In [ ]:
run_query('''
SELECT
    "Payment Method",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY "Payment Method"
ORDER BY churn_rate_pct DESC
''', "Churn by Payment Method")


## 6. Financial Queries


In [ ]:
run_query('''
SELECT
    CASE WHEN "Churn Value" = 1 THEN 'Churned' ELSE 'Retained' END AS status,
    ROUND(SUM("Monthly Charges"), 2) AS monthly_revenue,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly,
    ROUND(SUM("Total Charges"), 2) AS total_revenue,
    ROUND(AVG("Total Charges"), 2) AS avg_total
FROM customers
GROUP BY "Churn Value"
''', "Revenue at Risk from Churning Customers")


In [ ]:
run_query('''
SELECT
    CASE
        WHEN "Monthly Charges" < 30 THEN 'Low ($0-30)'
        WHEN "Monthly Charges" < 60 THEN 'Medium ($30-60)'
        WHEN "Monthly Charges" < 90 THEN 'High ($60-90)'
        ELSE 'Premium ($90+)'
    END AS charge_bracket,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY charge_bracket
ORDER BY charge_bracket
''', "Revenue by Charge Bracket")


## 7. Tenure & Loyalty Queries


In [ ]:
run_query('''
SELECT
    CASE
        WHEN "Tenure Months" <= 12 THEN '0-12 months'
        WHEN "Tenure Months" <= 24 THEN '12-24 months'
        WHEN "Tenure Months" <= 48 THEN '24-48 months'
        ELSE '48+ months'
    END AS tenure_cohort,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge
FROM customers
GROUP BY tenure_cohort
ORDER BY tenure_cohort
''', "Churn by Tenure Cohort")


## 8. Advanced Queries


In [ ]:
run_query('''
SELECT
    risk_factors,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM (
    SELECT *,
        (CASE WHEN "Contract" = 'Month-to-month' THEN 1 ELSE 0 END +
         CASE WHEN "Internet Service" = 'Fiber optic' THEN 1 ELSE 0 END +
         CASE WHEN "Payment Method" = 'Electronic check' THEN 1 ELSE 0 END +
         CASE WHEN "Tenure Months" <= 12 THEN 1 ELSE 0 END) AS risk_factors
    FROM customers
) subq
GROUP BY risk_factors
ORDER BY risk_factors
''', "Risk Factor Accumulation Analysis")


In [ ]:
run_query('''
SELECT
    "Contract",
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN "Monthly Charges" ELSE 0 END), 2) AS revenue_lost,
    ROUND(SUM("Monthly Charges"), 2) AS total_revenue,
    ROUND(
        SUM(CASE WHEN "Churn Value" = 1 THEN "Monthly Charges" ELSE 0 END) * 100.0 /
        SUM("Monthly Charges"),
    2) AS revenue_loss_pct
FROM customers
GROUP BY "Contract"
ORDER BY revenue_loss_pct DESC
''', "Monthly Revenue Churn Impact by Contract")


In [ ]:
run_query('''
SELECT
    "Internet Service",
    "Online Security",
    "Tech Support",
    COUNT(*) AS total,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
WHERE "Internet Service" != 'No'
GROUP BY "Internet Service", "Online Security", "Tech Support"
HAVING COUNT(*) >= 50
ORDER BY churn_rate_pct ASC
LIMIT 10
''', "Service Combinations with Best Retention")


In [ ]:
conn.close()
print("Database connection closed.")


## Summary

This notebook demonstrates SQL proficiency with 25+ business-oriented queries covering:
- Overview statistics and revenue analysis
- Demographic segmentation
- Service adoption impact
- Contract and payment risk factors
- Financial analysis and revenue at risk
- Tenure cohort analysis
- Advanced risk factor accumulation

All queries are also available in `sql/business_queries.sql`.
